# History Spotify User Silver 🥈
---

## Change logs

| Version | Description                | Update Date |
|---------|----------------------------|-------------|
| 1.0     | Silver Table completed     | 2026-06-23  |


## Import Libs

In [0]:
import pandas as pd
import glob
from dotenv import load_dotenv
import os
from pyspark.sql.functions import col,when,lower

## Load Env

In [0]:
load_dotenv()

## Read Bronze Table

In [0]:
df_audio_01 = spark.table(os.getenv("NAME_TABLE_AUDIO_BRONZE"))

In [0]:
display(df_audio_01.limit(5))

## Remove colluns not used

In [0]:
drop_columns = [
    'ip_addr',
    'spotify_track_uri',
    'spotify_episode_uri',
    'conn_country',
    'audiobook_uri',
    'audiobook_chapter_uri',
    'audiobook_chapter_uri',
    'offline',
    'offline_timestamp',
    'incognito_mode'
]

df_audio_02 = df_audio_01.drop(*drop_columns)
display(df_audio_02.limit(5))

## Change type and rename collumns

In [0]:
df_audio_03 = df_audio_02.select(
    col("ts").cast("timestamp").alias("ts_play"),
    col("platform").alias("nm_platform"),
    col("ms_played").alias("nr_ms_played"),
    col("master_metadata_track_name").alias("nm_track"),
    col("master_metadata_album_artist_name").alias("nm_artist"),
    col("master_metadata_album_album_name").alias("nm_album"),
    col("episode_name").alias("nm_episode"),
    col("episode_show_name").alias("nm_show"),
    col("audiobook_title").alias("nm_audiobook"),
    col("audiobook_chapter_title").alias("nm_audiobook_chapter"),
    col("reason_start").alias("nm_reason_start"),
    col("reason_end").alias("nm_reason_end"),
    col("shuffle").alias("fl_shuffle"),
    col("skipped").alias("fl_skipped")
)

In [0]:
display(df_audio_03.groupBy("nm_platform").count())

## Rename result columns

In [0]:
df_audio_04 = df_audio_03.withColumn(
    "nm_platform",
    when(
        lower(col("nm_platform")).contains("windows"), "Windows"
    ).when(
        lower(col("nm_platform")).contains("android"), "Android"
    ).when(
        lower(col("nm_platform")).contains("ios"), "IOS"
    ).when(
        lower(col("nm_platform")).contains("mac"), "Mac"
    ).when(
        lower(col("nm_platform")).contains("linux"), "Linux"
    ).when(
        lower(col("nm_platform")).contains("web"), "Web"
    ).when(
        lower(col("nm_platform")).contains("amazon"), "Alexa"
    ).otherwise("Not define")
)

## Save dataframe

In [0]:
df_audio_04.write.mode("overwrite").saveAsTable(os.getenv("NAME_TABLE_AUDIO_SILVER"))